# 00 · Audit of the Kaggle copy of STATS19 (why it was rejected)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam843/uk-road-accident-analytics/blob/main/notebooks/00_kaggle_data_audit.ipynb)

Earlier versions of this project used a Kaggle CSV that claimed to be the UK road accident data
for 2005-2014. This notebook checks that file against the official DfT release and records why
this project uses the **official DfT open data** instead.

**Input:** `DATA_DIR/kaggle/ukroadaccident.csv`. Supply it by hand; this repo never downloads
from Kaggle. If the file is absent, every cell below skips with a message.

Checks:
1. Rows per year: is any year missing?
2. Exact duplicate rows, especially in 2012, compared with the official 2012 total from RAS0101.
3. `Accident_Index` uniqueness, and indexes mangled into scientific notation (e.g. `2.01E+12`) by Excel.
4. Rows with missing or zero coordinates.
5. Codes that do not exist in the DfT data guide.

All numbers come from the cells below. Results: **TBD (run notebooks/00_kaggle_data_audit.ipynb)**.

In [ ]:
# Setup: works from the repo root, from notebooks/, or on Google Colab.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules and not Path("src/config.py").exists():
    if not Path("uk-road-accident-analytics").exists():
        !git clone -q https://github.com/maryam843/uk-road-accident-analytics.git
    %cd uk-road-accident-analytics
    %pip install -q -r requirements.txt

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import pandas as pd

from src import clean, config

KAGGLE_PATH = config.get_data_dir() / "kaggle" / "ukroadaccident.csv"
HAVE_KAGGLE = KAGGLE_PATH.exists()
if not HAVE_KAGGLE:
    print(f"Skipping audit: {KAGGLE_PATH} not found. Put the Kaggle CSV there to run this notebook.")

In [ ]:
def load_kaggle(path):
    """Read every column as text so nothing is silently converted."""
    df = pd.read_csv(path, dtype=str, low_memory=False, encoding_errors="replace")
    return clean.coalesce_duplicates(df.rename(columns=clean.build_column_map(df.columns)))


def kaggle_dates(df):
    """Parse the date column, trying ISO first and then DD/MM/YYYY."""
    iso = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
    dmy = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
    return iso.fillna(dmy)


if HAVE_KAGGLE:
    kaggle = load_kaggle(KAGGLE_PATH)
    kaggle["_date"] = kaggle_dates(kaggle)
    year_col = "year" if "year" in kaggle.columns else None
    kaggle["_year"] = (pd.to_numeric(kaggle[year_col], errors="coerce") if year_col
                       else kaggle["_date"].dt.year)
    print(f"{len(kaggle):,} rows, {kaggle.shape[1] - 2} columns")
    print("columns:", list(kaggle.columns[:-2]))

## 1. Rows per year
A year with 0 rows is missing from the file entirely.

In [ ]:
if HAVE_KAGGLE:
    years = range(int(kaggle["_year"].min()), int(kaggle["_year"].max()) + 1)
    rows_per_year = kaggle["_year"].value_counts().reindex(years, fill_value=0).rename("rows")
    display(rows_per_year.to_frame().T)
    print("Years with no rows:", [y for y, n in rows_per_year.items() if n == 0])

## 2. Exact duplicate rows
Compared with the official RAS0101 collision count for the same year (if `ras0101.ods` has been downloaded).

In [ ]:
if HAVE_KAGGLE:
    data_cols = [c for c in kaggle.columns if not c.startswith("_")]
    exact_dup = kaggle.duplicated(subset=data_cols, keep="first")
    dup_by_year = exact_dup.groupby(kaggle["_year"]).sum().rename("exact_duplicate_rows")
    summary = pd.concat([rows_per_year, dup_by_year], axis=1).fillna(0).astype(int)
    summary["rows_after_dedup"] = summary["rows"] - summary["exact_duplicate_rows"]
    ras_path = config.get_data_dir() / "raw" / config.RAS0101_RAW_FILE
    if ras_path.exists():
        official = clean.load_ras0101(ras_path)["collisions"]
        summary["official_ras0101"] = official.reindex(summary.index)
        summary["rows_minus_official"] = summary["rows"] - summary["official_ras0101"]
    display(summary)
    print(f"Exact duplicate rows in the whole file: {int(exact_dup.sum()):,}")
    if 2012 in summary.index:
        print(f"Exact duplicate rows in 2012: {int(summary.loc[2012, 'exact_duplicate_rows']):,}")

## 3. `Accident_Index` uniqueness and scientific notation

In [ ]:
if HAVE_KAGGLE:
    idx = kaggle["collision_index"].astype(str)
    sci = idx.str.fullmatch(r"\d+(\.\d+)?[eE][+-]?\d+")
    print(f"rows: {len(idx):,}")
    print(f"unique index values: {idx.nunique():,}")
    print(f"rows whose index is in scientific notation: {int(sci.sum()):,} ({sci.mean():.1%})")
    print("examples:", idx[sci].drop_duplicates().head(5).tolist())

## 4. Missing or zero coordinates

In [ ]:
if HAVE_KAGGLE:
    lat = pd.to_numeric(kaggle.get("latitude"), errors="coerce")
    lon = pd.to_numeric(kaggle.get("longitude"), errors="coerce")
    print(f"missing latitude or longitude: {int((lat.isna() | lon.isna()).sum()):,}")
    print(f"latitude exactly 0: {int((lat == 0).sum()):,}")
    outside = ~(lat.between(*config.GB_LAT_RANGE) & lon.between(*config.GB_LON_RANGE))
    print(f"outside the GB bounding box (incl. missing): {int(outside.sum()):,}")
    filled_mean = lat.round(4).value_counts().head(3)
    print("most frequent latitude values (a mean-imputed value shows up as a spike):")
    display(filled_mean.to_frame("rows"))

## 5. Codes not in the DfT data guide
Needs `DATA_DIR/raw/data_guide.xlsx`. Current and `_historic` code lists are both accepted.

In [ ]:
guide_path = config.get_data_dir() / "raw" / config.DATA_GUIDE_RAW_FILE
if HAVE_KAGGLE and guide_path.exists():
    codes = clean.load_code_list(guide_path)
    rows = []
    for col in clean.COLLISION_CODED_FIELDS:
        if col not in kaggle.columns:
            continue
        allowed = set(codes.loc[codes["field"].isin([col, f"{col}_historic"]), "code"])
        if col == "junction_detail_historic":
            allowed |= set(codes.loc[codes["field"] == "junction_detail", "code"])
        values = pd.to_numeric(kaggle[col], errors="coerce")
        text_values = kaggle[col].notna() & values.isna()
        odd = values.notna() & ~values.isin(allowed)
        rows.append({"column": col, "non_numeric": int(text_values.sum()),
                     "codes_not_in_guide": int(odd.sum()),
                     "examples": sorted(values[odd].unique().tolist())[:5]
                                 + kaggle.loc[text_values, col].unique().tolist()[:5]})
    display(pd.DataFrame(rows))
elif HAVE_KAGGLE:
    print("Data guide not found; run `python -m src.download --only guide` first.")

## Why this file was rejected

Each failure listed here is checked by a cell above. The size of each problem is whatever
those cells print.

- **Missing year:** a year with 0 rows breaks any trend, seasonality or forecasting work.
- **Duplicated rows:** exact copies inflate one year's count above the official total, which
  would show up as a fake spike in a forecast. They also put the same collision in both train
  and test.
- **Corrupted index:** values stored in scientific notation are no longer unique, so
  collisions cannot be joined to casualties, and duplicates cannot be found by index.
- **Imputed coordinates:** replacing missing coordinates with a mean or 0 puts collisions
  in the sea or at one fake point. Any map or spatial statistic built on that is wrong.

The official DfT release (`src/download.py`) has none of these problems by construction,
can be reconciled with published totals (notebook 01), and is licensed under OGL v3.0.